### QED-C Application-Oriented Benchmarks - Pre-defined Equal1-Runner

Using Qiskit to construct and run benchmarks, but inserting in Equal1 way of executing. 

## Default configurations
Change the following values to control benchmark parameters

In [4]:
# Generic becnhmark paramters
min_qubits = 2
max_qubits = 17
skip_qubits = 1
max_circuits = 2
num_shots = 1000

# Equal1 device specific
equal1_device = "equal1_simulator"
equal1_noise_model = "bell2-17-gen-preview"

Setup equal1 execution environment

In [5]:
import base64

from qbraid import QbraidProvider
from qiskit import QuantumCircuit, qasm2


class QBraidBackEnd():
    def __init__(self):
        self.name = "QBraidEqual1Backend"

class QBraidResult:
    def __init__(self, counts, exec_time, transpiled_circuit_metrics):
        self.exec_time = exec_time
        self.counts = counts
        self.transpiled_circuit_metrics = transpiled_circuit_metrics

    def get_counts(self, qc):
        return self.counts

    def get_transpiled_circuit_metrics(self):
        return self.transpiled_circuit_metrics

class QBraidExecutor():
    def __init__(self):
        provider = QbraidProvider()
        self.device = provider.get_device("equal1_simulator")

    def __call__(self, qc : QuantumCircuit, backend_name : str, backend, shots, **kwargs) -> QBraidResult:
        # print(f"attempting to run {qc} on {backend_name}, on {backend} with {shots} and {kwargs}")

        runtime_options = {
            "simulation_platform": "GPU",
            "execution_options": {"optimization_level": 2},
        }

        backend = "StateVector" if qc.num_qubits > 8 else "DensityMatrix"

        job = self.device.run(qc, shots=shots, noise_model="bell2-17-gen-preview", runtime_options=runtime_options, backend=backend)
        job.wait_for_final_state()  # Wait for the job to complete and get final state

        if job.status().name != "COMPLETED":
            job_result = job.client.get_job_results(job.id)
            print(f"\n@@@@@@@@@@@@@@ \n JOB FAILED for {qc} \n With error message:\n {job_result['statusText']}\n @@@@@@@@@@@@@@@@@@ \n ")


        result = job.result()
        counts = result.data.get_counts()
        # print(counts)

        result_json = job.client.get_job_results(job.id)
        inner_exec_time = result_json['executionMetrics']['executor']


        transpiled_circuit = base64.b64decode(result_json['compiledOutput']).decode('utf-8')
        transpiled_qc = qasm2.loads(transpiled_circuit, custom_instructions=qasm2.LEGACY_CUSTOM_INSTRUCTIONS)

        from _common.qiskit.execute import get_circuit_metrics
        metrics = get_circuit_metrics(transpiled_qc)


        return QBraidResult(counts, inner_exec_time, metrics)



In [6]:
# Common setup parameters
backend_id = "qasm_simulator"

hub = ""
group = ""
project = ""
provider_backend = None

exec_options = {
    "executor" : QBraidExecutor()
}


/home/iszilveszter/work/cuda-quantum/QC-App-Oriented-Benchmarks/.venv/lib64/python3.12/site-packages/qbraid_core/_compat.py:44: UserWarning: You are using qbraid-core version 0.1.41, however, version 0.2.0 is available. To avoid compatibility issues, consider upgrading.
  warnings.warn(
/home/iszilveszter/work/cuda-quantum/QC-App-Oriented-Benchmarks/.venv/lib64/python3.12/site-packages/qbraid/runtime/native/provider.py:151: RuntimeWarning: The default runtime configuration for device 'equal1_simulator' includes transpilation to program type 'cudaq', which is not registered.
  warnings.warn(


### Deutsch-Jozsa

In [ ]:
from deutsch_jozsa import dj_benchmark
dj_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

Deutsch-Jozsa Benchmark Program - Qiskit
... execution starting at Feb 23, 2026 18:16:41 UTC
************
Executing [2] circuits with num_qubits = 3



@@@@@@@@@@@@@@ 
 JOB FAILED for       ┌────────────┐               ░  ░ ┌────────────┐             ░ ┌─┐   
q0_0: ┤ U(π/2,0,π) ├───────────────░──░─┤ U(π/2,0,π) ├─────────────░─┤M├───
      ├────────────┤               ░  ░ ├────────────┤             ░ └╥┘┌─┐
q0_1: ┤ U(π/2,0,π) ├───────────────░──░─┤ U(π/2,0,π) ├─────────────░──╫─┤M├
      └┬──────────┬┘┌────────────┐ ░  ░ ├────────────┤┌──────────┐ ░  ║ └╥┘
q0_2: ─┤ U(π,0,π) ├─┤ U(π/2,0,π) ├─░──░─┤ U(π/2,0,π) ├┤ U(π,0,π) ├─░──╫──╫─
       └──────────┘ └────────────┘ ░  ░ └────────────┘└──────────┘ ░  ║  ║ 
c0: 2/════════════════════════════════════════════════════════════════╩══╩═
                                                                      0  1  
 With error message:
 'ERROR:  [Experiment 0] std::bad_alloc: cudaErrorDevicesUnavailable: CUDA-capable device(s) is/are busy or unavailable ,  ERROR: std::bad_alloc: cudaErrorDevicesUnavailable: CUDA-capable device(s) is/are busy or unavailable'
 @@@@@@@@@@@@@@@@@@ 
 
ERROR: Faile


@@@@@@@@@@@@@@ 
 JOB FAILED for       ┌────────────┐               ░ ┌───┐ ░            ░ ┌───┐ ░ »
q1_0: ┤ U(π/2,0,π) ├───────────────░─┤ X ├─░───■────────░─┤ X ├─░─»
      ├────────────┤               ░ └───┘ ░   │        ░ └───┘ ░ »
q1_1: ┤ U(π/2,0,π) ├───────────────░───────░───┼────■───░───────░─»
      └┬──────────┬┘┌────────────┐ ░       ░ ┌─┴─┐┌─┴─┐ ░       ░ »
q1_2: ─┤ U(π,0,π) ├─┤ U(π/2,0,π) ├─░───────░─┤ X ├┤ X ├─░───────░─»
       └──────────┘ └────────────┘ ░       ░ └───┘└───┘ ░       ░ »
c1: 2/════════════════════════════════════════════════════════════»
                                                                  »
«      ┌────────────┐             ░ ┌─┐   
«q1_0: ┤ U(π/2,0,π) ├─────────────░─┤M├───
«      ├────────────┤             ░ └╥┘┌─┐
«q1_1: ┤ U(π/2,0,π) ├─────────────░──╫─┤M├
«      ├────────────┤┌──────────┐ ░  ║ └╥┘
«q1_2: ┤ U(π/2,0,π) ├┤ U(π,0,π) ├─░──╫──╫─
«      └────────────┘└──────────┘ ░  ║  ║ 
«c1: 2/══════════════════════════════╩══╩═
«          


@@@@@@@@@@@@@@ 
 JOB FAILED for       ┌────────────┐               ░ ┌───┐ ░                      ░ ┌───┐ ░ »
q5_0: ┤ U(π/2,0,π) ├───────────────░─┤ X ├─░───■──────────────────░─┤ X ├─░─»
      ├────────────┤               ░ └───┘ ░   │                  ░ └───┘ ░ »
q5_1: ┤ U(π/2,0,π) ├───────────────░───────░───┼────■─────────────░───────░─»
      ├────────────┤               ░ ┌───┐ ░   │    │             ░ ┌───┐ ░ »
q5_2: ┤ U(π/2,0,π) ├───────────────░─┤ X ├─░───┼────┼────■────────░─┤ X ├─░─»
      ├────────────┤               ░ └───┘ ░   │    │    │        ░ └───┘ ░ »
q5_3: ┤ U(π/2,0,π) ├───────────────░───────░───┼────┼────┼────■───░───────░─»
      └┬──────────┬┘┌────────────┐ ░       ░ ┌─┴─┐┌─┴─┐┌─┴─┐┌─┴─┐ ░       ░ »
q5_4: ─┤ U(π,0,π) ├─┤ U(π/2,0,π) ├─░───────░─┤ X ├┤ X ├┤ X ├┤ X ├─░───────░─»
       └──────────┘ └────────────┘ ░       ░ └───┘└───┘└───┘└───┘ ░       ░ »
c5: 4/══════════════════════════════════════════════════════════════════════»
                               


@@@@@@@@@@@@@@ 
 JOB FAILED for       ┌────────────┐               ░ ┌───┐ ░                                ░ »
q9_0: ┤ U(π/2,0,π) ├───────────────░─┤ X ├─░───■────────────────────────────░─»
      ├────────────┤               ░ └───┘ ░   │                            ░ »
q9_1: ┤ U(π/2,0,π) ├───────────────░───────░───┼────■───────────────────────░─»
      ├────────────┤               ░ ┌───┐ ░   │    │                       ░ »
q9_2: ┤ U(π/2,0,π) ├───────────────░─┤ X ├─░───┼────┼────■──────────────────░─»
      ├────────────┤               ░ └───┘ ░   │    │    │                  ░ »
q9_3: ┤ U(π/2,0,π) ├───────────────░───────░───┼────┼────┼────■─────────────░─»
      ├────────────┤               ░ ┌───┐ ░   │    │    │    │             ░ »
q9_4: ┤ U(π/2,0,π) ├───────────────░─┤ X ├─░───┼────┼────┼────┼────■────────░─»
      ├────────────┤               ░ └───┘ ░   │    │    │    │    │        ░ »
q9_5: ┤ U(π/2,0,π) ├───────────────░───────░───┼────┼────┼────┼────┼────■───░─»
      └

### Bernstein-Vazirani - Method 1

In [ ]:
import sys
sys.path.insert(1, "bernstein-vazirani")
import bv_benchmark
bv_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                method=1,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Hidden Shift

In [ ]:
import sys
sys.path.insert(1, "hidden-shift")
import hs_benchmark
hs_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Grover

In [ ]:
import sys
sys.path.insert(1, "grovers/qiskit")
import grovers_benchmark

grover_max_circuits = 8 if max_circuits > 8 else max_circuits
grovers_benchmark.run(min_qubits=8, max_qubits=8, skip_qubits=skip_qubits,
                max_circuits=grover_max_circuits, num_shots=num_shots,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Phase Estimation

In [ ]:
import sys
sys.path.insert(1, "phase-estimation")
import pe_benchmark
pe_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### HHL Linear Solver

In [ ]:
import sys
sys.path.insert(1, "hhl/qiskit")
import hhl_benchmark

hhl_benchmark.verbose=False

hhl_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                method=1, use_best_widths=True,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Amplitude Estimation

In [ ]:
import sys
sys.path.insert(1, "amplitude-estimation/qiskit")


import ae_benchmark
ae_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)



### Monte Carlo

In [ ]:
import sys
sys.path.insert(1, "monte-carlo/qiskit")
import mc_benchmark
mc_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Hamiltonian Simulation - Method 1 

In [ ]:
import sys
sys.path.insert(1, "hamiltonian-simulation/qiskit")
import hamiltonian_simulation_benchmark
hamiltonian_simulation_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                method=1,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Hamiltonian Simulation - Method 2 

In [ ]:
import sys
sys.path.insert(1, "hamiltonian-simulation/qiskit")
import hamiltonian_simulation_benchmark
hamiltonian_simulation_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                method=2,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### VQE - Method 1

In [ ]:
import sys
sys.path.insert(1, "vqe/qiskit")
import vqe_benchmark
vqe_num_shots=4098
vqe_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits,
                max_circuits=max_circuits, num_shots=vqe_num_shots,
                method=1,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Shor - Method 1

In [ ]:
import sys
sys.path.insert(1, "shors/qiskit")
import shors_benchmark
shors_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, max_circuits=1, num_shots=num_shots,
                method=1,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Shor - Method 2

In [ ]:
import sys
sys.path.insert(1, "shors/qiskit")
import shors_benchmark
shors_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, max_circuits=1, num_shots=num_shots,
                method=2,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Quantum Fourier Transform - Method 1

In [ ]:
import sys
sys.path.insert(1, "quantum-fourier-transform")
import qft_benchmark
qft_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                method=1,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Quantum Fourier Transform - Method 2

In [ ]:
import sys
sys.path.insert(1, "quantum-fourier-transform")
import qft_benchmark
qft_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                method=2,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Bernstein-Vazirani - Method 2

In [ ]:
import sys
sys.path.insert(1, "bernstein-vazirani")
import bv_benchmark
bv_benchmark.run(min_qubits=min_qubits, max_qubits=max_qubits, skip_qubits=skip_qubits,
                max_circuits=max_circuits, num_shots=num_shots,
                method=2,
                backend_id=backend_id, provider_backend=provider_backend,
                hub=hub, group=group, project=project, exec_options=exec_options)

### Close session
IMPORTANT: This cell is provided as a way to close an active session if for some reason the benchmarks abort abnormally.
If so, execute the close_session function manually to terminate an open session. Normally, this is done automatically.

In [ ]:
import sys
sys.path.insert(1, "_common")
import execute as ex

ex.close_session()

In [ ]:
import sys
sys.path.insert(1, "_common")
import metrics

# metrics.depth_base = 2
# metrics.QV = 0
# apps = [ "Hidden Shift", "Grover's Search", "Quantum Fourier Transform (1)", "Hamiltonian Simulation" ]
# backend_id='qasm_simulator'

metrics.plot_all_app_metrics(backend_id, do_all_plots=False, include_apps=None)